# Chemical Organization of the Molecular Embedding Space

## Setup

In [ ]:
from setup import *

import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
from torch.utils.data import random_split

## Compute Motif Projections

In [ ]:
def load_cached_molecules(dataset, cache):
    loader = DataLoader(dataset, batch_size=64, collate_fn=ConformerEdge_collate_fn, shuffle=False, pin_memory=True)
    lookup = {value.item(): index for index, value in enumerate(cache["id"])}
    embeddings, ids, properties = [], [], {}
    excluded_attrs = {"coord", "numbers", "charges", "charge", "smiles", "edge_index", "edge_attr", "ptr", "batch", "fingerprint"}
    for batch in tqdm(loader):
        for key, data in batch.items():
            if key in excluded_attrs: continue
            properties.setdefault(key, []).append(data.cpu())
        molecule_ids = batch.smiles.cpu()
        indices = torch.tensor([lookup[value.item()] for value in molecule_ids])
        embeddings.append(cache["embeddings"][indices])
        ids.append(molecule_ids)
    return (torch.cat(embeddings).cpu().numpy(), {key: torch.cat(values).numpy() for key, values in properties.items()}, torch.cat(ids).numpy())

def colorize(properties, color_map, special_colorization):
    colors = {}
    for name in properties:
        if name in special_colorization: 
            min_value, max_value = special_colorization[name]
            colors[name] = color_map((np.clip(properties[name], min_value, max_value) - min_value) / (max_value - min_value))
        else: colors[name] = color_map(np.clip(properties[name], 0, 4) / (4.0 - 0.0))
    return colors

def sample_grid_molecules(coordinates, ids, id_to_smiles, n_bins=10, samples_per_cell=20, seed=0):
    rng = random.Random(seed)
    ids = np.asarray(ids)
    assert len(ids) == len(coordinates)
    x, y = coordinates[:, 0], coordinates[:, 1]
    x_edges = np.linspace(x.min(), x.max(), n_bins + 1)
    y_edges = np.linspace(y.min(), y.max(), n_bins + 1)
    x_idx = np.clip(np.digitize(x, x_edges) - 1, 0, n_bins - 1)
    y_idx = np.clip(np.digitize(y, y_edges) - 1, 0, n_bins - 1)
    cells = {}
    for index, cell in enumerate(zip(x_idx, y_idx)): cells.setdefault(cell, []).append(index)
    return {
        cell: [(int(ids[index]), id_to_smiles.get(int(ids[index]), f"<id={int(ids[index])}, not found>"))
               for index in rng.sample(indices, min(samples_per_cell, len(indices)))]
        for cell, indices in sorted(cells.items())
    }

def prepare_motif_projection(name):
    dataset = ConformerEdgeDataset(str(CHEMBL_DIR), num_conformers=16, sample_per_batch=1, use_molecule_properties=True)
    generator = torch.Generator().manual_seed(0)
    torch.manual_seed(0)
    random.seed(0)
    np.random.seed(0)
    _, dataset = random_split(dataset, [0.95, 0.05], generator=generator)
    cache = load_or_compute_embeddings(EMBEDDING_DIR / f"embeddings_{name}.pt", CHECKPOINTS[name], CHEMBL_DIR, DEVICE)
    embeddings, properties, ids_used = load_cached_molecules(dataset, cache)
    colors = colorize(properties, plt.get_cmap("Spectral"), {"weight": (100.0, 600.0)})
    print(f"{name}: computing t-SNE")
    tsne_data = compute_tsne(embeddings, seed=0, perplexity=300)
    print(f"{name}: computing PCA")
    pca_data = compute_pca(embeddings)
    return {"tsne": tsne_data, "pca": pca_data, "ids": ids_used, "colors": colors}

motif_results = {name: prepare_motif_projection(name) for name in ("ANT", "NT", "nopre")}
id_to_smiles = load_smiles_lookup(str(SMILES_DIR))
grid_samples = {
    model_name: {projection: sample_grid_molecules(data[projection], data["ids"], id_to_smiles=id_to_smiles) for projection in ("tsne", "pca")}
    for model_name, data in motif_results.items()
}

## Figures 2 and S2–S6: Organization by Chemical Motifs

In [ ]:
def plot_motif_panel(model_name, motif, projection="tsne"):
    data = motif_results[model_name]
    coordinates = data[projection]
    fig, ax = plt.subplots(figsize=(5 * cm, 5 * cm), constrained_layout=True)
    fig.set_constrained_layout_pads(w_pad=0, h_pad=0, wspace=0, hspace=0)
    for spine in ax.spines.values(): spine.set_linewidth(0.25)
    ax.scatter(coordinates[:, 0], coordinates[:, 1], c=data["colors"][motif], s=0.3, edgecolors="none")
    ax.set_position([0.01, 0.01, 0.98, 0.98])
    ax.set_xticks([])
    ax.set_yticks([])
    label = "tSNE" if projection == "tsne" else "PCA"
    fig.savefig(OUTPUT_DIR / f"ChEMBL36_{label}_{motif}_{model_name}.png", dpi=600)
    plt.show()
    plt.close(fig)

for model_name, data in motif_results.items():
    for projection in ("tsne", "pca"):
        for motif in data["colors"]:
            plot_motif_panel(model_name, motif, projection=projection)

color_map = plt.get_cmap("Spectral")
count_values = [0, 1, 2, 3, 4]
count_labels = ["0", "1", "2", "3", "4+"]
swatch_colors = [color_map(v / 4.0) for v in count_values]
fig_w, fig_h = 1.6 * cm, 6 * cm
fig, ax = plt.subplots(figsize=(fig_w, fig_h))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_aspect(fig_h / fig_w)  # keeps 1 unit of x equal to 1 unit of y physically -> true squares

border = FancyBboxPatch(
    (0.01, 0.01), 0.98, 0.98,   # smaller offset from axes edge (was 0.03, 0.02)
    boxstyle="round,pad=0",
    linewidth=0.5, edgecolor="black", facecolor="white",
    transform=ax.transAxes, zorder=0,
)
ax.add_patch(border)
ax.text(0.5, 0.93, "count", fontsize=9, ha="center", va="center", transform=ax.transAxes)
n = len(count_values)
row_height = 0.9 / n
top_y = 0.9
swatch_frac_of_height = row_height * 0.55
swatch_w = swatch_frac_of_height * (fig_h / fig_w)
swatch_h = swatch_frac_of_height

for i, (color, label) in enumerate(zip(swatch_colors, count_labels)):
    row_center_y = top_y - row_height * (i + 0.5)
    swatch = mpatches.Rectangle(
        (0.12, row_center_y - swatch_h / 2), swatch_w, swatch_h,
        facecolor=color, edgecolor="none", transform=ax.transAxes,
    )
    ax.add_patch(swatch)
    ax.text(0.12 + swatch_w + 0.15, row_center_y, label,
            fontsize=9, ha="left", va="center", transform=ax.transAxes)

plt.savefig(OUTPUT_DIR / "legend.png", dpi=600, bbox_inches="tight", pad_inches=0.02)
plt.show()

## Grid-Sampled Molecules

In [ ]:
display(pd.DataFrame([
    {"Model": model_name, "Projection": projection, "Grid cell": cell,
     "ID": molecule_id, "SMILES": smiles}
    for model_name, projections in grid_samples.items()
    for projection, cells in projections.items()
    for cell, molecules in cells.items()
    for molecule_id, smiles in molecules
]))